# MediTwin AI — Hospital Operations EDA (Phase 2)

**Educational prototype only.** Synthetic hospital data — not for clinical use.

This notebook explores cleaned + feature-engineered hospital operational data.
Run the pipeline first from the project root:

```bash
python -m src.features.feature_engineering
```

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Ensure project root is on path when notebook is opened from /notebooks
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.preprocess import (
    load_hospital_data,
    preprocess_hospital_data,
    validate_hospital_processed,
)
from src.features.feature_engineering import (
    engineer_hospital_features,
    validate_engineered_hospital,
)

pd.set_option("display.max_columns", 50)
print("Project root:", PROJECT_ROOT)

## 1. Load, preprocess, engineer features

In [ ]:
feat_path = PROJECT_ROOT / "data" / "processed" / "hospital_operations_features.csv"

if feat_path.is_file():
    df = pd.read_csv(feat_path, parse_dates=["date"])
    print("Loaded processed features from", feat_path)
else:
    raw = load_hospital_data()
    clean, report = preprocess_hospital_data(raw)
    report.print_report()
    df = engineer_hospital_features(clean)

df = df.sort_values("date").reset_index(drop=True)
print(df.shape)
df.head()

In [ ]:
errors = validate_engineered_hospital(df)
print("Validation errors:", errors if errors else "None")
df.describe().T

## 2. OPD and emergency trends over time

In [ ]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=("OPD patients over time", "Emergency patients over time"))
fig.add_trace(go.Scatter(x=df["date"], y=df["opd_patients"], name="OPD",
                         line=dict(color="#1f77b4")), row=1, col=1)
if "rolling_7_day_opd_mean" in df.columns:
    fig.add_trace(go.Scatter(x=df["date"], y=df["rolling_7_day_opd_mean"],
                             name="7-day OPD mean (past)",
                             line=dict(color="#ff7f0e", dash="dash")), row=1, col=1)
fig.add_trace(go.Scatter(x=df["date"], y=df["emergency_patients"], name="Emergency",
                         line=dict(color="#d62728")), row=2, col=1)
fig.update_layout(height=600, title_text="Patient volume trends", legend_tracegroupgap=0)
fig.show()

## 3. Bed and ICU occupancy

In [ ]:
fig = px.line(df, x="date", y=["bed_occupancy_percentage", "icu_occupancy_percentage"],
              title="Bed vs ICU occupancy (%)")
fig.add_hline(y=90, line_dash="dot", annotation_text="90% stress threshold")
fig.show()

print("Bed occupancy mean:", round(df["bed_occupancy_percentage"].mean(), 2))
print("ICU occupancy mean:", round(df["icu_occupancy_percentage"].mean(), 2))
print("Days with peak_load_indicator=1:", int(df["peak_load_indicator"].sum()))

## 4. Waiting time and medicine consumption

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("Waiting time (min)", "Medicine consumption"))
fig.add_trace(go.Scatter(x=df["date"], y=df["average_waiting_time_minutes"],
                         name="Wait", line=dict(color="#9467bd")), row=1, col=1)
fig.add_trace(go.Scatter(x=df["date"], y=df["medicine_consumption_units"],
                         name="Medicine", line=dict(color="#2ca02c")), row=1, col=2)
fig.update_layout(height=400, title_text="Service pressure indicators", showlegend=False)
fig.show()

corr_wait_opd = df["average_waiting_time_minutes"].corr(df["opd_patients"])
corr_med_load = df["medicine_consumption_units"].corr(df["total_patient_load"])
print(f"Corr(wait, OPD) = {corr_wait_opd:.3f}")
print(f"Corr(medicine, total_patient_load) = {corr_med_load:.3f}")

## 5. Correlation heatmap (key numeric features)

In [ ]:
corr_cols = [
    "opd_patients", "emergency_patients", "admissions", "discharges",
    "bed_occupancy_percentage", "icu_occupancy_percentage",
    "average_waiting_time_minutes", "medicine_consumption_units",
    "doctors_available", "staff_to_patient_ratio", "emergency_ratio",
    "daily_patient_count", "total_patient_load",
]
corr_cols = [c for c in corr_cols if c in df.columns]
corr = df[corr_cols].corr()

fig = px.imshow(corr, text_auto=".2f", aspect="auto",
                title="Correlation matrix (hospital features)",
                color_continuous_scale="RdBu_r", zmin=-1, zmax=1)
fig.update_layout(height=700)
fig.show()

## 6. Weekday patterns

In [ ]:
day_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
by_dow = df.groupby("day_of_week")[["opd_patients", "emergency_patients",
                                      "average_waiting_time_minutes",
                                      "bed_occupancy_percentage"]].mean()
by_dow.index = [day_names[i] for i in by_dow.index]

fig = px.bar(by_dow.reset_index().rename(columns={"index": "weekday"}),
             x="weekday", y="opd_patients",
             title="Average OPD by weekday")
fig.show()

fig2 = px.bar(by_dow.reset_index().rename(columns={"index": "weekday"}),
              x="weekday", y="emergency_patients",
              title="Average emergency volume by weekday")
fig2.show()

weekend_opd = df.loc[df["is_weekend"] == 1, "opd_patients"].mean()
weekday_opd = df.loc[df["is_weekend"] == 0, "opd_patients"].mean()
print(f"Mean OPD weekday={weekday_opd:.1f}, weekend={weekend_opd:.1f}")

## 7. Monthly / seasonal patterns

In [ ]:
by_month = df.groupby("month")[["opd_patients", "emergency_patients",
                                 "icu_occupancy_percentage"]].mean().reset_index()
fig = px.line(by_month, x="month", y=["opd_patients", "emergency_patients"],
              markers=True, title="Monthly average patient volumes")
fig.show()

by_season = df.groupby("season")[["opd_patients", "emergency_patients",
                                   "bed_occupancy_percentage",
                                   "icu_occupancy_percentage"]].mean()
display(by_season.round(2))

fig = px.box(df, x="season", y="opd_patients",
             title="OPD distribution by season")
fig.show()

## 8. Matplotlib snapshot — occupancy vs waiting time

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(df["bed_occupancy_percentage"], df["average_waiting_time_minutes"],
                c=df["opd_patients"], cmap="viridis", alpha=0.7)
ax.set_xlabel("Bed occupancy %")
ax.set_ylabel("Average waiting time (min)")
ax.set_title("Occupancy vs waiting time (colour = OPD)")
plt.colorbar(sc, ax=ax, label="OPD patients")
plt.tight_layout()
plt.show()

## 9. Quick pattern notes (fill after running)

Document observations here for the viva:
- Weekend vs weekday OPD
- Seasonal peaks
- Strongest correlations
- Spike / peak-load days